# Second-last latent probes

This notebook asks when a causal transformer represents either of the two latents that are fully determined by the 26-token input and are needed to predict token 27:

1. **Second-last level-1 latent:** node 7 of 9, spanning tokens 22–24. Probe the residual stream at the boundary after token **24** (Python index `23`).
2. **Second-last level-2 latent:** node 1 of 3, spanning tokens 10–18. Probe the residual stream at the boundary after token **18** (Python index `17`).

Positions in the prose are 1-based; tensor indices are 0-based. The train and test examples are generated independently, are fully ternary (27 terminals), and the transformer sees only the first 26 terminals. A separate linear probe is fit at the embedding and after every transformer block.


## 1. Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

WORKSPACE = Path("/Users/parley/Documents/Last_token")
sys.path.insert(0, str(WORKSPACE))
sys.path.insert(0, str(WORKSPACE / "datasets" / "rhm"))

import models
from mixed_random_hierarchy_model import sample_padded_rules

torch.manual_seed(0)
np.random.seed(0)

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print("PyTorch", torch.__version__, "| device:", DEVICE)

In [ ]:
# Grammar: these values must match the training run.
V = 8
L = 3
BIN = 2
SEED_RULES = 12345678

# Model: num_heads cannot be inferred from the checkpoint tensors.
NUM_HEADS = 16
CHECKPOINT = WORKSPACE / "run_L3_v8_bin2_P16777216_lr0.001_l20.03_bs128_arr0_t1024.pt"

# Probe data and classifier.
N_TRAIN = 22_000
N_TEST = 10_000
TRAIN_SEED = 42
TEST_SEED = 43
ACTIVATION_BATCH_SIZE = 256
PROBE_C = 0.1

M_2 = BIN
M_3 = BIN * V
D_MAX = 3 ** L             # 27 terminals
CONTEXT_LENGTH = D_MAX - 1 # 26 tokens supplied to the model

# User-facing positions are 1-based boundaries. Convert once, explicitly.
LEVEL1_BOUNDARY = 24
LEVEL2_BOUNDARY = 18
LEVEL1_POSITION = LEVEL1_BOUNDARY - 1  # tensor index 23
LEVEL2_POSITION = LEVEL2_BOUNDARY - 1  # tensor index 17
LEVEL1_NODE = 7                         # second-last of 9 nodes
LEVEL2_NODE = 1                         # second-last of 3 nodes

assert LEVEL1_POSITION == 23 and LEVEL1_NODE == 7
assert LEVEL2_POSITION == 17 and LEVEL2_NODE == 1
assert CHECKPOINT.exists(), f"Checkpoint not found: {CHECKPOINT}"

print(f"Full sequence: {D_MAX}; model input: {CONTEXT_LENGTH}")
print(f"Level-1 target node {LEVEL1_NODE}, activation index {LEVEL1_POSITION}")
print(f"Level-2 target node {LEVEL2_NODE}, activation index {LEVEL2_POSITION}")

## 2. Fully ternary probing train/test sets

The grammar is reconstructed once from the training seed. Train and test sequences use different sampling seeds. `latents[0]` is the root, `latents[1]` contains the three level-2 nodes, and `latents[2]` contains the nine level-1 nodes.


In [ ]:
rules = sample_padded_rules(V, V, M_2, M_3, 2, 3, L, SEED_RULES)

# All mass is assigned to the ternary rules at every grammar level.
ternary_probs = {
    level: torch.cat([torch.zeros(M_2), torch.full((M_3,), 1.0 / M_3)])
    for level in range(L)
}


def generate_fully_ternary(n_examples, seed):
    """Return terminals (N, 27) and latents [root, level-2, level-1]."""
    generator = torch.Generator().manual_seed(seed)
    root = torch.randint(0, V, (n_examples,), generator=generator)
    features = root.clone()
    latents = [root.clone()]

    for level in range(L - 1):
        flat = features.reshape(-1)
        selected = torch.multinomial(
            ternary_probs[level].expand(flat.numel(), -1),
            num_samples=1,
            replacement=True,
            generator=generator,
        ).squeeze(-1).reshape(features.shape)
        features = rules[level][features, selected].flatten(start_dim=1)
        latents.append(features.clone())

    flat = features.reshape(-1)
    selected = torch.multinomial(
        ternary_probs[L - 1].expand(flat.numel(), -1),
        num_samples=1,
        replacement=True,
        generator=generator,
    ).squeeze(-1).reshape(features.shape)
    terminals = rules[L - 1][features, selected].flatten(start_dim=1)
    return terminals.long(), latents


train_terminals, train_latents = generate_fully_ternary(N_TRAIN, TRAIN_SEED)
test_terminals, test_latents = generate_fully_ternary(N_TEST, TEST_SEED)

train_inputs = train_terminals[:, :CONTEXT_LENGTH]
test_inputs = test_terminals[:, :CONTEXT_LENGTH]

assert train_terminals.shape == (N_TRAIN, 27)
assert test_terminals.shape == (N_TEST, 27)
assert train_inputs.shape == (N_TRAIN, 26)
assert test_inputs.shape == (N_TEST, 26)

print("train terminals / inputs:", train_terminals.shape, train_inputs.shape)
print("test terminals / inputs: ", test_terminals.shape, test_inputs.shape)
print("latent shapes:", [tuple(x.shape) for x in train_latents])

## 3. Load the trained causal transformer

In [ ]:
def infer_architecture(state_dict, num_heads):
    embedding_dim = state_dict["token_embedding_table.weight"].shape[1]
    num_layers = max(
        int(key.split(".")[1]) for key in state_dict if key.startswith("blocks.")
    ) + 1
    return dict(
        vocab_size=state_dict["token_embedding_table.weight"].shape[0],
        block_size=state_dict["blocks.0.attn.tril"].shape[0],
        embedding_dim=embedding_dim,
        num_heads=num_heads,
        ffwd_size=state_dict["blocks.0.ffwd.net.0.weight"].shape[0] // embedding_dim,
        num_layers=num_layers,
        dropout=0.0,
        rope="position_embedding_table.weight" not in state_dict,
        share_emb=False,
    )


def load_model(checkpoint_path, num_heads, device):
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    architecture = infer_architecture(checkpoint["model"], num_heads)
    model = models.CLM(**architecture)
    model.load_state_dict(checkpoint["model"])
    model.to(device).eval()
    return model, architecture, checkpoint.get("step"), checkpoint.get("state", {})


model, architecture, checkpoint_step, checkpoint_state = load_model(
    CHECKPOINT, NUM_HEADS, DEVICE
)
NUM_LAYERS = architecture["num_layers"]
print("architecture:", architecture)
print("step:", checkpoint_step, "| val_loss:", checkpoint_state.get("val_loss", "?"))

## 4. Extract one position across all layers

Only the requested causal position is retained, rather than materializing activations for all 26 positions. Probe point 0 is the token-plus-position embedding; probe point `k` is the residual stream after transformer block `k`.


In [ ]:
@torch.inference_mode()
def extract_position_activations(model, inputs, position, batch_size=256):
    """Return (embedding + blocks, examples, hidden_dim) at one position."""
    if not 0 <= position < inputs.shape[1]:
        raise ValueError(f"position {position} outside sequence length {inputs.shape[1]}")

    per_layer = [[] for _ in range(model.num_layers + 1)]

    for start in range(0, len(inputs), batch_size):
        batch = inputs[start:start + batch_size].to(DEVICE)
        captured = {}

        pre_hook = model.blocks[0].register_forward_pre_hook(
            lambda module, args: captured.__setitem__(0, args[0][:, position].detach().cpu())
        )

        def save_block(layer_index):
            def hook(module, args, output):
                captured[layer_index + 1] = output[:, position].detach().cpu()
            return hook

        block_hooks = [
            block.register_forward_hook(save_block(i))
            for i, block in enumerate(model.blocks)
        ]
        try:
            model(batch)
        finally:
            pre_hook.remove()
            for hook in block_hooks:
                hook.remove()

        for layer_index in range(model.num_layers + 1):
            per_layer[layer_index].append(captured[layer_index])

    return np.stack([
        torch.cat(batches, dim=0).numpy() for batches in per_layer
    ])


def fit_layerwise_probes(train_activations, train_targets,
                         test_activations, test_targets, C=0.1):
    """Fit independent standardized multinomial linear probes at every layer."""
    if train_activations.shape[0] != test_activations.shape[0]:
        raise ValueError("Train and test activations have different layer counts")

    train_accuracy, test_accuracy = [], []
    for layer_index, (x_train, x_test) in enumerate(
        zip(train_activations, test_activations)
    ):
        probe = Pipeline([
            ("scale", StandardScaler()),
            ("classifier", LogisticRegression(
                C=C, max_iter=5000, solver="lbfgs", random_state=0
            )),
        ])
        probe.fit(x_train, train_targets)
        train_score = probe.score(x_train, train_targets)
        test_score = probe.score(x_test, test_targets)
        train_accuracy.append(train_score)
        test_accuracy.append(test_score)
        print(f"layer {layer_index:2d}: train={train_score:.3f}, test={test_score:.3f}")

    return {"train": np.asarray(train_accuracy), "test": np.asarray(test_accuracy)}


def plot_layerwise_accuracy(result, title):
    layer_names = ["emb"] + [f"L{i}" for i in range(1, NUM_LAYERS + 1)]
    x = np.arange(len(layer_names))
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(x, result["train"], "o--", alpha=0.55, label="train")
    ax.plot(x, result["test"], "o-", linewidth=2.3, label="test")
    ax.axhline(1 / V, color="gray", linestyle=":", label=f"chance = 1/{V}")
    ax.set(xticks=x, xticklabels=layer_names, ylim=(0, 1.02),
           xlabel="Probe point", ylabel="Linear-probe accuracy", title=title)
    ax.grid(alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()

## Part A — second-last level-1 latent

The label is `latents[2][:, 7]`, the second-last of the nine level-1 nodes. It is fully fixed by tokens 22–24, so the causal residual stream is read immediately after token 24 (tensor index 23).


In [ ]:
level1_train_targets = train_latents[L - 1][:, LEVEL1_NODE].numpy()
level1_test_targets = test_latents[L - 1][:, LEVEL1_NODE].numpy()

print("train class counts:", np.bincount(level1_train_targets, minlength=V))
print("test class counts: ", np.bincount(level1_test_targets, minlength=V))

level1_train_activations = extract_position_activations(
    model, train_inputs, LEVEL1_POSITION, ACTIVATION_BATCH_SIZE
)
level1_test_activations = extract_position_activations(
    model, test_inputs, LEVEL1_POSITION, ACTIVATION_BATCH_SIZE
)
print("train/test activations:", level1_train_activations.shape,
      level1_test_activations.shape)

In [ ]:
level1_results = fit_layerwise_probes(
    level1_train_activations, level1_train_targets,
    level1_test_activations, level1_test_targets,
    C=PROBE_C,
)
plot_layerwise_accuracy(
    level1_results,
    "Second-last level-1 latent — activation after token 24",
)

## Part B — second-last level-2 latent

The label is `latents[1][:, 1]`, the middle (and therefore second-last) of the three level-2 nodes. It is fully fixed by tokens 10–18, so the causal residual stream is read immediately after token 18 (tensor index 17).


In [ ]:
level2_train_targets = train_latents[L - 2][:, LEVEL2_NODE].numpy()
level2_test_targets = test_latents[L - 2][:, LEVEL2_NODE].numpy()

print("train class counts:", np.bincount(level2_train_targets, minlength=V))
print("test class counts: ", np.bincount(level2_test_targets, minlength=V))

level2_train_activations = extract_position_activations(
    model, train_inputs, LEVEL2_POSITION, ACTIVATION_BATCH_SIZE
)
level2_test_activations = extract_position_activations(
    model, test_inputs, LEVEL2_POSITION, ACTIVATION_BATCH_SIZE
)
print("train/test activations:", level2_train_activations.shape,
      level2_test_activations.shape)

In [ ]:
level2_results = fit_layerwise_probes(
    level2_train_activations, level2_train_targets,
    level2_test_activations, level2_test_targets,
    C=PROBE_C,
)
plot_layerwise_accuracy(
    level2_results,
    "Second-last level-2 latent — activation after token 18",
)

## Optional comparison

This final plot puts the two held-out accuracy curves on the same axes. The two probes still use their own targets and their own causal positions.


In [ ]:
layer_names = ["emb"] + [f"L{i}" for i in range(1, NUM_LAYERS + 1)]
x = np.arange(len(layer_names))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x, level1_results["test"], "o-", linewidth=2.3,
        label="level-1 node 7 @ token 24")
ax.plot(x, level2_results["test"], "s-", linewidth=2.3,
        label="level-2 node 1 @ token 18")
ax.axhline(1 / V, color="gray", linestyle=":", label=f"chance = 1/{V}")
ax.set(xticks=x, xticklabels=layer_names, ylim=(0, 1.02),
       xlabel="Probe point", ylabel="Held-out linear-probe accuracy",
       title="Second-last latent information across transformer layers")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

# Extension — third-last latent probes

The second-last analysis above is left intact. This extension repeats it for the preceding fully observed subtree:

1. **Third-last level-1 latent:** node 6 of 9, spanning tokens 19–21. Read the causal residual stream after token **21** (Python index `20`).
2. **Third-last level-2 latent:** node 0 of 3, spanning the first 9 tokens. Read the causal residual stream after token **9** (Python index `8`).

These targets are also completely determined by the available prefix at their respective probe positions.


In [ ]:
THIRD_LAST_LEVEL1_NODE = 6
THIRD_LAST_LEVEL2_NODE = 0

THIRD_LAST_LEVEL1_BOUNDARY = 21
THIRD_LAST_LEVEL2_BOUNDARY = 9
THIRD_LAST_LEVEL1_POSITION = THIRD_LAST_LEVEL1_BOUNDARY - 1  # tensor index 20
THIRD_LAST_LEVEL2_POSITION = THIRD_LAST_LEVEL2_BOUNDARY - 1  # tensor index 8

assert THIRD_LAST_LEVEL1_NODE == 6 and THIRD_LAST_LEVEL1_POSITION == 20
assert THIRD_LAST_LEVEL2_NODE == 0 and THIRD_LAST_LEVEL2_POSITION == 8

print(
    f"Third-last level-1: node {THIRD_LAST_LEVEL1_NODE}, "
    f"activation after token {THIRD_LAST_LEVEL1_BOUNDARY} "
    f"(index {THIRD_LAST_LEVEL1_POSITION})"
)
print(
    f"Third-last level-2: node {THIRD_LAST_LEVEL2_NODE}, "
    f"activation after token {THIRD_LAST_LEVEL2_BOUNDARY} "
    f"(index {THIRD_LAST_LEVEL2_POSITION})"
)

## Part C — third-last level-1 latent

The target is `latents[2][:, 6]`, whose three terminal children are tokens 19–21. The activation is taken at the final token of that span.


In [ ]:
third_level1_train_targets = train_latents[L - 1][:, THIRD_LAST_LEVEL1_NODE].numpy()
third_level1_test_targets = test_latents[L - 1][:, THIRD_LAST_LEVEL1_NODE].numpy()

print("train class counts:", np.bincount(third_level1_train_targets, minlength=V))
print("test class counts: ", np.bincount(third_level1_test_targets, minlength=V))

third_level1_train_activations = extract_position_activations(
    model, train_inputs, THIRD_LAST_LEVEL1_POSITION, ACTIVATION_BATCH_SIZE
)
third_level1_test_activations = extract_position_activations(
    model, test_inputs, THIRD_LAST_LEVEL1_POSITION, ACTIVATION_BATCH_SIZE
)
print(
    "train/test activations:",
    third_level1_train_activations.shape,
    third_level1_test_activations.shape,
)

In [ ]:
third_level1_results = fit_layerwise_probes(
    third_level1_train_activations, third_level1_train_targets,
    third_level1_test_activations, third_level1_test_targets,
    C=PROBE_C,
)
plot_layerwise_accuracy(
    third_level1_results,
    "Third-last level-1 latent — activation after token 21",
)

## Part D — third-last level-2 latent

The target is `latents[1][:, 0]`, whose nine terminal descendants are the first nine tokens. The activation is taken at token 9, before any information from the central or final level-2 subtree is available.


In [ ]:
third_level2_train_targets = train_latents[L - 2][:, THIRD_LAST_LEVEL2_NODE].numpy()
third_level2_test_targets = test_latents[L - 2][:, THIRD_LAST_LEVEL2_NODE].numpy()

print("train class counts:", np.bincount(third_level2_train_targets, minlength=V))
print("test class counts: ", np.bincount(third_level2_test_targets, minlength=V))

third_level2_train_activations = extract_position_activations(
    model, train_inputs, THIRD_LAST_LEVEL2_POSITION, ACTIVATION_BATCH_SIZE
)
third_level2_test_activations = extract_position_activations(
    model, test_inputs, THIRD_LAST_LEVEL2_POSITION, ACTIVATION_BATCH_SIZE
)
print(
    "train/test activations:",
    third_level2_train_activations.shape,
    third_level2_test_activations.shape,
)

In [ ]:
third_level2_results = fit_layerwise_probes(
    third_level2_train_activations, third_level2_train_targets,
    third_level2_test_activations, third_level2_test_targets,
    C=PROBE_C,
)
plot_layerwise_accuracy(
    third_level2_results,
    "Third-last level-2 latent — activation after token 9",
)

## Third-last comparison

This plot compares held-out accuracy for the two new probes. Each curve uses its own latent target and the causal position at the end of its corresponding subtree.


In [ ]:
layer_names = ["emb"] + [f"L{i}" for i in range(1, NUM_LAYERS + 1)]
x = np.arange(len(layer_names))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x, third_level1_results["test"], "o-", linewidth=2.3,
        label="level-1 node 6 @ token 21")
ax.plot(x, third_level2_results["test"], "s-", linewidth=2.3,
        label="level-2 node 0 @ token 9")
ax.axhline(1 / V, color="gray", linestyle=":", label=f"chance = 1/{V}")
ax.set(
    xticks=x,
    xticklabels=layer_names,
    ylim=(0, 1.02),
    xlabel="Probe point",
    ylabel="Held-out linear-probe accuracy",
    title="Third-last latent information across transformer layers",
)
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

# Final four-panel checkpoint comparison

This final cell runs all four probes at checkpoints `t=5792`, `t=262144`, and `t=1048576`. It extracts the four positions together for efficiency, fits layerwise probes using the shared train/test split, saves all numerical results and metadata to a pickle file, and saves the final 2×2 plot as both PNG and PDF.

Panel layout:

| | Level-1 | Level-2 |
|---|---|---|
| **Second-last** | node 7 @ token 24 | node 1 @ token 18 |
| **Third-last** | node 6 @ token 21 | node 0 @ token 9 |


In [ ]:
# This is deliberately self-contained at the experiment level: running this cell
# regenerates every curve in the final figure from the already-created probe sets.
import gc
import pickle

FINAL_CHECKPOINTS = {
    512: WORKSPACE / "run_L3_v8_bin2_P16777216_lr0.001_l20.03_bs128_arr0_t512.pt",
    262_144: WORKSPACE / "run_L3_v8_bin2_P16777216_lr0.001_l20.03_bs128_arr0_t262144.pt",
    1_048_576: WORKSPACE / "run_L3_v8_bin2_P16777216_lr0.001_l20.03_bs128_arr0_t1048576.pt",
}

FINAL_PROBES = {
    "second_last_level1": {
        "row": 0, "col": 0, "level": 1,
        "node": LEVEL1_NODE, "boundary": LEVEL1_BOUNDARY,
        "position": LEVEL1_POSITION,
        "train_targets": level1_train_targets,
        "test_targets": level1_test_targets,
    },
    "second_last_level2": {
        "row": 0, "col": 1, "level": 2,
        "node": LEVEL2_NODE, "boundary": LEVEL2_BOUNDARY,
        "position": LEVEL2_POSITION,
        "train_targets": level2_train_targets,
        "test_targets": level2_test_targets,
    },
    "third_last_level1": {
        "row": 1, "col": 0, "level": 1,
        "node": THIRD_LAST_LEVEL1_NODE, "boundary": THIRD_LAST_LEVEL1_BOUNDARY,
        "position": THIRD_LAST_LEVEL1_POSITION,
        "train_targets": third_level1_train_targets,
        "test_targets": third_level1_test_targets,
    },
    "third_last_level2": {
        "row": 1, "col": 1, "level": 2,
        "node": THIRD_LAST_LEVEL2_NODE, "boundary": THIRD_LAST_LEVEL2_BOUNDARY,
        "position": THIRD_LAST_LEVEL2_POSITION,
        "train_targets": third_level2_train_targets,
        "test_targets": third_level2_test_targets,
    },
}

missing = [str(path) for path in FINAL_CHECKPOINTS.values() if not path.exists()]
assert not missing, "Missing checkpoints:\n" + "\n".join(missing)


@torch.inference_mode()
def extract_multiple_position_activations(model, inputs, positions, batch_size=256):
    """Extract several causal positions in one forward pass.

    Returns a dict mapping each probe name to an array shaped
    (embedding + blocks, examples, hidden_dim).
    """
    for name, position in positions.items():
        if not 0 <= position < inputs.shape[1]:
            raise ValueError(f"{name}: position {position} outside length {inputs.shape[1]}")

    collected = {
        name: [[] for _ in range(model.num_layers + 1)]
        for name in positions
    }

    for start in range(0, len(inputs), batch_size):
        batch = inputs[start:start + batch_size].to(DEVICE)
        captured = {name: {} for name in positions}

        def capture_embedding(module, args):
            embedding = args[0]
            for name, position in positions.items():
                captured[name][0] = embedding[:, position].detach().cpu()

        pre_hook = model.blocks[0].register_forward_pre_hook(capture_embedding)

        def capture_block(layer_index):
            def hook(module, args, output):
                for name, position in positions.items():
                    captured[name][layer_index + 1] = output[:, position].detach().cpu()
            return hook

        block_hooks = [
            block.register_forward_hook(capture_block(layer_index))
            for layer_index, block in enumerate(model.blocks)
        ]
        try:
            model(batch)
        finally:
            pre_hook.remove()
            for hook in block_hooks:
                hook.remove()

        for name in positions:
            for layer_index in range(model.num_layers + 1):
                collected[name][layer_index].append(captured[name][layer_index])

    return {
        name: np.stack([
            torch.cat(layer_batches, dim=0).numpy()
            for layer_batches in collected[name]
        ])
        for name in positions
    }


positions = {name: config["position"] for name, config in FINAL_PROBES.items()}
final_results = {}
final_architecture = None

for requested_step, checkpoint_path in FINAL_CHECKPOINTS.items():
    print(f"\n{'=' * 72}\nCheckpoint t={requested_step}: {checkpoint_path.name}")
    checkpoint_model, checkpoint_arch, saved_step, saved_state = load_model(
        checkpoint_path, NUM_HEADS, DEVICE
    )
    if saved_step is not None:
        assert int(saved_step) == requested_step, (
            f"Filename requests t={requested_step}, checkpoint stores step={saved_step}"
        )
    if final_architecture is None:
        final_architecture = checkpoint_arch
    else:
        assert checkpoint_arch == final_architecture, "Checkpoint architectures differ"

    print("Extracting train activations at all four positions ...")
    train_acts = extract_multiple_position_activations(
        checkpoint_model, train_inputs, positions, ACTIVATION_BATCH_SIZE
    )
    print("Extracting test activations at all four positions ...")
    test_acts = extract_multiple_position_activations(
        checkpoint_model, test_inputs, positions, ACTIVATION_BATCH_SIZE
    )

    final_results[requested_step] = {
        "checkpoint": str(checkpoint_path),
        "saved_step": saved_step,
        "val_loss": saved_state.get("val_loss"),
        "probes": {},
    }
    for probe_name, config in FINAL_PROBES.items():
        print(f"\n{probe_name}")
        final_results[requested_step]["probes"][probe_name] = fit_layerwise_probes(
            train_acts[probe_name], config["train_targets"],
            test_acts[probe_name], config["test_targets"],
            C=PROBE_C,
        )

    del train_acts, test_acts, checkpoint_model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE == "mps":
        torch.mps.empty_cache()


final_layer_labels = ["emb"] + [
    f"L{i}" for i in range(1, final_architecture["num_layers"] + 1)
]

# Keep targets out of the saved metadata; the bundle contains everything needed
# to reproduce the figure without retaining the large generated datasets.
saved_probe_metadata = {
    name: {key: value for key, value in config.items()
           if key not in {"train_targets", "test_targets"}}
    for name, config in FINAL_PROBES.items()
}
final_plot_data = {
    "results": final_results,
    "probe_metadata": saved_probe_metadata,
    "layer_labels": final_layer_labels,
    "chance_accuracy": 1 / V,
    "architecture": final_architecture,
    "grammar": {"V": V, "L": L, "BIN": BIN, "SEED_RULES": SEED_RULES},
    "dataset": {
        "n_train": N_TRAIN, "n_test": N_TEST,
        "train_seed": TRAIN_SEED, "test_seed": TEST_SEED,
        "full_length": D_MAX, "input_length": CONTEXT_LENGTH,
    },
    "probe_C": PROBE_C,
}

RESULTS_PATH = WORKSPACE / "second_third_last_probe_checkpoint_results_new.pkl"
with RESULTS_PATH.open("wb") as handle:
    pickle.dump(final_plot_data, handle)
print(f"\nSaved plot data to {RESULTS_PATH}")


# 2×2 layout: rows select second-/third-last; columns select level 1/2.
panel_titles = {
    "second_last_level1": "Second-last level-1 latent (token 24)",
    "second_last_level2": "Second-last level-2 latent (token 18)",
    "third_last_level1": "Third-last level-1 latent (token 21)",
    "third_last_level2": "Third-last level-2 latent (token 9)",
}
colors = {512: "tab:blue", 262_144: "tab:orange", 1_048_576: "tab:green"}
x = np.arange(len(final_layer_labels))
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)

for probe_name, config in FINAL_PROBES.items():
    ax = axes[config["row"], config["col"]]
    for step in FINAL_CHECKPOINTS:
        accuracy = final_results[step]["probes"][probe_name]["test"]
        ax.plot(x, accuracy, "o-", linewidth=2.2, markersize=6,
                color=colors[step], label=f"t={step:,}")
    ax.axhline(1 / V, color="gray", linestyle=":", linewidth=1.5,
               label=f"chance = 1/{V}")
    ax.set_title(panel_titles[probe_name])
    ax.set_xticks(x)
    ax.set_xticklabels(final_layer_labels)
    ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.25)

for ax in axes[:, 0]:
    ax.set_ylabel("Held-out linear-probe accuracy")
for ax in axes[1, :]:
    ax.set_xlabel("Probe point")

# One shared legend keeps all four panels visually comparable.
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4, frameon=False,
           bbox_to_anchor=(0.5, 0.99))
fig.suptitle("Second-last and third-last latent information across training",
             fontsize=15, y=1.02)
fig.tight_layout(rect=(0, 0, 1, 0.94))

PNG_PATH = WORKSPACE / "second_third_last_probe_checkpoint_comparison_new.png"
PDF_PATH = WORKSPACE / "second_third_last_probe_checkpoint_comparison_new.pdf"
fig.savefig(PNG_PATH, dpi=200, bbox_inches="tight")
fig.savefig(PDF_PATH, bbox_inches="tight")
print(f"Saved figure to {PNG_PATH}")
print(f"Saved figure to {PDF_PATH}")
plt.show()

In [ ]:
# 2×2 layout: rows select second-/third-last; columns select level 1/2.
panel_titles = {
    "second_last_level1": "Second-last level-1 latent (token 24)",
    "second_last_level2": "Second-last level-2 latent (token 18)",
    "third_last_level1": "Third-last level-1 latent (token 21)",
    "third_last_level2": "Third-last level-2 latent (token 9)",
}
colors = {1_024: "tab:blue", 262_144: "tab:orange", 1_048_576: "tab:green"}
x = np.arange(len(final_layer_labels))
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)

for probe_name, config in FINAL_PROBES.items():
    ax = axes[config["row"], config["col"]]
    for step in FINAL_CHECKPOINTS:
        accuracy = final_results[step]["probes"][probe_name]["test"]
        ax.plot(x, accuracy, "o-", linewidth=2.2, markersize=6,
                color=colors[step], label=f"t={step:,}")
    ax.axhline(1 / V, color="gray", linestyle=":", linewidth=1.5,
               label=f"chance = 1/{V}")
    ax.set_title(panel_titles[probe_name])
    ax.set_xticks(x)
    ax.set_xticklabels(final_layer_labels)
    ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.25)

for ax in axes[:, 0]:
    ax.set_ylabel("Held-out linear-probe accuracy")
for ax in axes[1, :]:
    ax.set_xlabel("Probe point")

# One shared legend keeps all four panels visually comparable.
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4, frameon=False,
           bbox_to_anchor=(0.5, 0.99))
fig.suptitle("Second-last and third-last latent information across training",
             fontsize=15, y=1.02)
fig.tight_layout(rect=(0, 0, 1, 0.94))

PNG_PATH = WORKSPACE / "second_third_last_probe_checkpoint_comparison_new.png"
PDF_PATH = WORKSPACE / "second_third_last_probe_checkpoint_comparison_new.pdf"
fig.savefig(PNG_PATH, dpi=200, bbox_inches="tight")
fig.savefig(PDF_PATH, bbox_inches="tight")
print(f"Saved figure to {PNG_PATH}")
print(f"Saved figure to {PDF_PATH}")
plt.show()

In [ ]:
# This is deliberately self-contained at the experiment level: running this cell
# regenerates every curve in the final figure from the already-created probe sets.
import gc
import pickle

FINAL_CHECKPOINTS = {
    512: WORKSPACE / "run_L3_v8_bin2_P16777216_lr0.001_l20.03_bs128_arr0_t512.pt",
    5_792: WORKSPACE / "run_L3_v8_bin2_P16777216_lr0.001_l20.03_bs128_arr0_t5792.pt",
    262_144: WORKSPACE / "run_L3_v8_bin2_P16777216_lr0.001_l20.03_bs128_arr0_t262144.pt",
    1_048_576: WORKSPACE / "run_L3_v8_bin2_P16777216_lr0.001_l20.03_bs128_arr0_t1048576.pt",
}

FINAL_PROBES = {
    "second_last_level1": {
        "row": 0, "col": 0, "level": 1,
        "node": LEVEL1_NODE, "boundary": LEVEL1_BOUNDARY,
        "position": LEVEL1_POSITION,
        "train_targets": level1_train_targets,
        "test_targets": level1_test_targets,
    },
    "second_last_level2": {
        "row": 0, "col": 1, "level": 2,
        "node": LEVEL2_NODE, "boundary": LEVEL2_BOUNDARY,
        "position": LEVEL2_POSITION,
        "train_targets": level2_train_targets,
        "test_targets": level2_test_targets,
    },
    "third_last_level1": {
        "row": 1, "col": 0, "level": 1,
        "node": THIRD_LAST_LEVEL1_NODE, "boundary": THIRD_LAST_LEVEL1_BOUNDARY,
        "position": THIRD_LAST_LEVEL1_POSITION,
        "train_targets": third_level1_train_targets,
        "test_targets": third_level1_test_targets,
    },
    "third_last_level2": {
        "row": 1, "col": 1, "level": 2,
        "node": THIRD_LAST_LEVEL2_NODE, "boundary": THIRD_LAST_LEVEL2_BOUNDARY,
        "position": THIRD_LAST_LEVEL2_POSITION,
        "train_targets": third_level2_train_targets,
        "test_targets": third_level2_test_targets,
    },
}

missing = [str(path) for path in FINAL_CHECKPOINTS.values() if not path.exists()]
assert not missing, "Missing checkpoints:\n" + "\n".join(missing)


@torch.inference_mode()
def extract_multiple_position_activations(model, inputs, positions, batch_size=256):
    """Extract several causal positions in one forward pass.

    Returns a dict mapping each probe name to an array shaped
    (embedding + blocks, examples, hidden_dim).
    """
    for name, position in positions.items():
        if not 0 <= position < inputs.shape[1]:
            raise ValueError(f"{name}: position {position} outside length {inputs.shape[1]}")

    collected = {
        name: [[] for _ in range(model.num_layers + 1)]
        for name in positions
    }

    for start in range(0, len(inputs), batch_size):
        batch = inputs[start:start + batch_size].to(DEVICE)
        captured = {name: {} for name in positions}

        def capture_embedding(module, args):
            embedding = args[0]
            for name, position in positions.items():
                captured[name][0] = embedding[:, position].detach().cpu()

        pre_hook = model.blocks[0].register_forward_pre_hook(capture_embedding)

        def capture_block(layer_index):
            def hook(module, args, output):
                for name, position in positions.items():
                    captured[name][layer_index + 1] = output[:, position].detach().cpu()
            return hook

        block_hooks = [
            block.register_forward_hook(capture_block(layer_index))
            for layer_index, block in enumerate(model.blocks)
        ]
        try:
            model(batch)
        finally:
            pre_hook.remove()
            for hook in block_hooks:
                hook.remove()

        for name in positions:
            for layer_index in range(model.num_layers + 1):
                collected[name][layer_index].append(captured[name][layer_index])

    return {
        name: np.stack([
            torch.cat(layer_batches, dim=0).numpy()
            for layer_batches in collected[name]
        ])
        for name in positions
    }


positions = {name: config["position"] for name, config in FINAL_PROBES.items()}
final_results = {}
final_architecture = None

for requested_step, checkpoint_path in FINAL_CHECKPOINTS.items():
    print(f"\n{'=' * 72}\nCheckpoint t={requested_step}: {checkpoint_path.name}")
    checkpoint_model, checkpoint_arch, saved_step, saved_state = load_model(
        checkpoint_path, NUM_HEADS, DEVICE
    )
    if saved_step is not None:
        assert int(saved_step) == requested_step, (
            f"Filename requests t={requested_step}, checkpoint stores step={saved_step}"
        )
    if final_architecture is None:
        final_architecture = checkpoint_arch
    else:
        assert checkpoint_arch == final_architecture, "Checkpoint architectures differ"

    print("Extracting train activations at all four positions ...")
    train_acts = extract_multiple_position_activations(
        checkpoint_model, train_inputs, positions, ACTIVATION_BATCH_SIZE
    )
    print("Extracting test activations at all four positions ...")
    test_acts = extract_multiple_position_activations(
        checkpoint_model, test_inputs, positions, ACTIVATION_BATCH_SIZE
    )

    final_results[requested_step] = {
        "checkpoint": str(checkpoint_path),
        "saved_step": saved_step,
        "val_loss": saved_state.get("val_loss"),
        "probes": {},
    }
    for probe_name, config in FINAL_PROBES.items():
        print(f"\n{probe_name}")
        final_results[requested_step]["probes"][probe_name] = fit_layerwise_probes(
            train_acts[probe_name], config["train_targets"],
            test_acts[probe_name], config["test_targets"],
            C=PROBE_C,
        )

    del train_acts, test_acts, checkpoint_model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE == "mps":
        torch.mps.empty_cache()


final_layer_labels = ["emb"] + [
    f"L{i}" for i in range(1, final_architecture["num_layers"] + 1)
]

# Keep targets out of the saved metadata; the bundle contains everything needed
# to reproduce the figure without retaining the large generated datasets.
saved_probe_metadata = {
    name: {key: value for key, value in config.items()
           if key not in {"train_targets", "test_targets"}}
    for name, config in FINAL_PROBES.items()
}
final_plot_data = {
    "results": final_results,
    "probe_metadata": saved_probe_metadata,
    "layer_labels": final_layer_labels,
    "chance_accuracy": 1 / V,
    "architecture": final_architecture,
    "grammar": {"V": V, "L": L, "BIN": BIN, "SEED_RULES": SEED_RULES},
    "dataset": {
        "n_train": N_TRAIN, "n_test": N_TEST,
        "train_seed": TRAIN_SEED, "test_seed": TEST_SEED,
        "full_length": D_MAX, "input_length": CONTEXT_LENGTH,
    },
    "probe_C": PROBE_C,
}

RESULTS_PATH = WORKSPACE / "second_third_last_probe_checkpoint_results.pkl"
with RESULTS_PATH.open("wb") as handle:
    pickle.dump(final_plot_data, handle)
print(f"\nSaved plot data to {RESULTS_PATH}")


# 2×2 layout: rows select second-/third-last; columns select level 1/2.
panel_titles = {
    "second_last_level1": "Second-last level-1 latent (token 24)",
    "second_last_level2": "Second-last level-2 latent (token 18)",
    "third_last_level1": "Third-last level-1 latent (token 21)",
    "third_last_level2": "Third-last level-2 latent (token 9)",
}
colors = {512: "tab:blue", 5_792: "tab:orange", 262_144: "tab:green", 1_048_576: "tab:red"}
x = np.arange(len(final_layer_labels))
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)

for probe_name, config in FINAL_PROBES.items():
    ax = axes[config["row"], config["col"]]
    for step in FINAL_CHECKPOINTS:
        accuracy = final_results[step]["probes"][probe_name]["test"]
        ax.plot(x, accuracy, "o-", linewidth=2.2, markersize=6,
                color=colors[step], label=f"t={step:,}")
    ax.axhline(1 / V, color="gray", linestyle=":", linewidth=1.5,
               label=f"chance = 1/{V}")
    ax.set_title(panel_titles[probe_name])
    ax.set_xticks(x)
    ax.set_xticklabels(final_layer_labels)
    ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.25)

for ax in axes[:, 0]:
    ax.set_ylabel("Held-out linear-probe accuracy")
for ax in axes[1, :]:
    ax.set_xlabel("Probe point")

# One shared legend keeps all four panels visually comparable.
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4, frameon=False,
           bbox_to_anchor=(0.5, 0.99))
fig.suptitle("Second-last and third-last latent information across training",
             fontsize=15, y=1.02)
fig.tight_layout(rect=(0, 0, 1, 0.94))

PNG_PATH = WORKSPACE / "second_third_last_probe_checkpoint_comparison.png"
PDF_PATH = WORKSPACE / "second_third_last_probe_checkpoint_comparison.pdf"
fig.savefig(PNG_PATH, dpi=200, bbox_inches="tight")
fig.savefig(PDF_PATH, bbox_inches="tight")
print(f"Saved figure to {PNG_PATH}")
print(f"Saved figure to {PDF_PATH}")
plt.show()

In [ ]:
# ---------------------------------------------------------------------
# Summary plot: L3 probe accuracy across training stages
# ---------------------------------------------------------------------

STAGE_STEPS = [512, 5_792, 262_144, 1_048_576]
STAGE_LABELS = ["Stage 1", "Stage 2", "Stage 3", "Stage 4"]

SUMMARY_PROBES = {
    "second_last_level1": "Second-last level-1 latent",
    "third_last_level1": "Third-last level-1 latent",
    "third_last_level2": "Third-last level-2 latent",
}

SUMMARY_COLORS = {
    "second_last_level1": "tab:purple",
    "third_last_level1": "tab:brown",
    "third_last_level2": "tab:cyan",
}

# Layer labels are ["emb", "L1", "L2", "L3", ...]
L3_INDEX = final_layer_labels.index("L3")

x = np.arange(len(STAGE_STEPS))

fig, ax = plt.subplots(figsize=(7.5, 5.0))

for probe_name, label in SUMMARY_PROBES.items():
    accuracies = [
        final_results[step]["probes"][probe_name]["test"][L3_INDEX]
        for step in STAGE_STEPS
    ]

    ax.plot(
        x,
        accuracies,
        marker="o",
        linewidth=2.2,
        markersize=7,
        color=SUMMARY_COLORS[probe_name],
        label=label,
    )

ax.axhline(
    1 / V,
    color="gray",
    linestyle=":",
    linewidth=1.5,
    label=f"Chance = 1/{V}",
)

ax.set_xticks(x)
ax.set_xticklabels(STAGE_LABELS)

ax.set_xlabel("Training stage")
ax.set_ylabel("Probe accuracy at L3")

ax.set_ylim(0, 1.02)
ax.grid(alpha=0.25)

ax.legend(frameon=False)

fig.tight_layout()

PNG_PATH = WORKSPACE / "latent_summary_across_stages.png"
PDF_PATH = WORKSPACE / "latent_summary_across_stages.pdf"

fig.savefig(PNG_PATH, dpi=200, bbox_inches="tight")
fig.savefig(PDF_PATH, bbox_inches="tight")

print(f"Saved figure to {PNG_PATH}")
print(f"Saved figure to {PDF_PATH}")

plt.show()

In [ ]:
STAGE_STEPS = [512, 5_792, 262_144, 1_048_576]
STAGE_LABELS = ["Stage 1", "Stage 2", "Stage 3", "Stage 4"]

SUMMARY_PROBES = {
    "second_last_level1": "Second-last level-1 latent",
    "third_last_level1": "Third-last level-1 latent",
    "third_last_level2": "Third-last level-2 latent",
}

SUMMARY_COLORS = {
    "second_last_level1": "tab:purple",
    "third_last_level1": "tab:brown",
    "third_last_level2": "tab:cyan",
}

# Layer labels are ["emb", "L1", "L2", "L3", ...]
L3_INDEX = final_layer_labels.index("L3")

x = np.arange(len(STAGE_STEPS))
bar_width = 0.24

fig, ax = plt.subplots(figsize=(7.5, 5.0))

for i, (probe_name, label) in enumerate(SUMMARY_PROBES.items()):
    accuracies = [
        final_results[step]["probes"][probe_name]["test"][L3_INDEX]
        for step in STAGE_STEPS
    ]

    offset = (i - 1) * bar_width

    ax.bar(
        x + offset,
        accuracies,
        width=bar_width,
        color=SUMMARY_COLORS[probe_name],
        label=label,
    )

ax.axhline(
    1 / V,
    color="gray",
    linestyle=":",
    linewidth=1.5,
    label=f"Chance = 1/{V}",
)

ax.set_xticks(x)
ax.set_xticklabels(STAGE_LABELS)

ax.set_xlabel("Training stage")
ax.set_ylabel("Probe accuracy at L3")

ax.set_ylim(0, 1.02)
ax.grid(axis="y", alpha=0.25)

ax.legend(frameon=False)

fig.tight_layout()

plt.show()

In [ ]:
from matplotlib.lines import Line2D

# ---------------------------------------------------------------------
# Three-panel summary figure
#
# (a) Level-1 latents vs transformer depth
# (b) Third-last level-2 latent vs transformer depth
# (c) L3 probe accuracy across training stages
# ---------------------------------------------------------------------

STAGE_STEPS = [512, 5_792, 262_144, 1_048_576]
STAGE_LABELS = ["Stage 1", "Stage 2", "Stage 3", "Stage 4"]

STEP_COLORS = {
    512: "tab:blue",
    5_792: "tab:orange",
    262_144: "tab:green",
    1_048_576: "tab:red",
}

LATENT_COLORS = {
    "second_last_level1": "tab:purple",
    "third_last_level1": "tab:brown",
    "third_last_level2": "tab:cyan",
}

LATENT_LABELS = {
    "second_last_level1": "Second-last level-1",
    "third_last_level1": "Third-last level-1",
    "third_last_level2": "Third-last level-2",
}

# Different line styles distinguish the two level-1 latents in panel (a).
LATENT_STYLES = {
    "second_last_level1": {
        "linestyle": "-",
        "marker": "o",
    },
    "third_last_level1": {
        "linestyle": "--",
        "marker": "s",
    },
}

x_depth = np.arange(len(final_layer_labels))
x_stage = np.arange(len(STAGE_STEPS))

L3_INDEX = final_layer_labels.index("L3")

# ---------------------------------------------------------------------
# Figure
# ---------------------------------------------------------------------

fig, axes = plt.subplots(
    1,
    3,
    figsize=(14.0, 4.2),
)

ax1, ax2, ax3 = axes


# =====================================================================
# (a) Level-1 latents across transformer depth
# =====================================================================

for step in STAGE_STEPS:
    for probe_name in ["second_last_level1", "third_last_level1"]:
        accuracy = final_results[step]["probes"][probe_name]["test"]

        ax1.plot(
            x_depth,
            accuracy,
            color=STEP_COLORS[step],
            linestyle=LATENT_STYLES[probe_name]["linestyle"],
            marker=LATENT_STYLES[probe_name]["marker"],
            linewidth=2.0,
            markersize=5,
        )

ax1.axhline(
    1 / V,
    color="gray",
    linestyle=":",
    linewidth=1.3,
)

ax1.set_xticks(x_depth)
ax1.set_xticklabels(final_layer_labels)

ax1.set_xlabel("Probe point")
ax1.set_ylabel("Held-out probe accuracy")

ax1.set_ylim(0, 1.02)
ax1.grid(alpha=0.2)

ax1.set_title("(a) Level-1 latents")


# =====================================================================
# (b) Third-last level-2 latent across transformer depth
# =====================================================================

probe_name = "third_last_level2"

for step in STAGE_STEPS:
    accuracy = final_results[step]["probes"][probe_name]["test"]

    ax2.plot(
        x_depth,
        accuracy,
        "o-",
        color=STEP_COLORS[step],
        linewidth=2.0,
        markersize=5,
    )

ax2.axhline(
    1 / V,
    color="gray",
    linestyle=":",
    linewidth=1.3,
)

ax2.set_xticks(x_depth)
ax2.set_xticklabels(final_layer_labels)

ax2.set_xlabel("Probe point")

ax2.set_ylim(0, 1.02)
ax2.grid(alpha=0.2)

ax2.set_title("(b) Third-last level-2 latent")


# =====================================================================
# (c) L3 accuracy across training stages
# =====================================================================

BAR_PROBES = [
    "second_last_level1",
    "third_last_level1",
    "third_last_level2",
]

bar_width = 0.23

for i, probe_name in enumerate(BAR_PROBES):

    accuracies = [
        final_results[step]["probes"][probe_name]["test"][L3_INDEX]
        for step in STAGE_STEPS
    ]

    offset = (i - 1) * bar_width

    ax3.bar(
        x_stage + offset,
        accuracies,
        width=bar_width,
        color=LATENT_COLORS[probe_name],
        label=LATENT_LABELS[probe_name],
    )

ax3.axhline(
    1 / V,
    color="gray",
    linestyle=":",
    linewidth=1.3,
)

ax3.set_xticks(x_stage)
ax3.set_xticklabels(STAGE_LABELS)

ax3.set_xlabel("Training stage")

ax3.set_ylim(0, 1.02)
ax3.grid(axis="y", alpha=0.2)

ax3.set_title("(c) Accuracy at L3")


# =====================================================================
# Shared legends
# =====================================================================

# Legend for checkpoint / training stage colors
stage_handles = [
    Line2D(
        [0],
        [0],
        color=STEP_COLORS[step],
        marker="o",
        linewidth=2,
        label=f"Stage {i + 1}",
    )
    for i, step in enumerate(STAGE_STEPS)
]

# Legend for the two line styles in panel (a)
level1_handles = [
    Line2D(
        [0],
        [0],
        color="black",
        linestyle=LATENT_STYLES[probe_name]["linestyle"],
        marker=LATENT_STYLES[probe_name]["marker"],
        linewidth=2,
        label=LATENT_LABELS[probe_name],
    )
    for probe_name in ["second_last_level1", "third_last_level1"]
]

# Put compact legends inside panels
legend1 = ax1.legend(
    handles=stage_handles,
    loc="lower right",
    frameon=False,
    fontsize=9,
)

ax1.add_artist(legend1)

ax1.legend(
    handles=level1_handles,
    loc="upper left",
    frameon=False,
    fontsize=9,
)

ax3.legend(
    frameon=False,
    fontsize=9,
)


# ---------------------------------------------------------------------
# Final layout
# ---------------------------------------------------------------------

fig.tight_layout(w_pad=1.6)

PDF_PATH = WORKSPACE / "latent_probe_three_panel_summary.pdf"
PNG_PATH = WORKSPACE / "latent_probe_three_panel_summary.png"

fig.savefig(PDF_PATH, bbox_inches="tight")
fig.savefig(PNG_PATH, dpi=300, bbox_inches="tight")

print(f"Saved figure to {PDF_PATH}")
print(f"Saved figure to {PNG_PATH}")

plt.show()

In [ ]:
# This is deliberately self-contained at the experiment level: running this cell
# regenerates every curve in the final figure from the already-created probe sets.
import gc
import pickle

FINAL_CHECKPOINTS = {
    512: WORKSPACE / "ternary_checkpoints/run_L3_v8_bin2_P16777216_lr0.001_l20.03_bs128_arr0_t512.pt",
    5_792: WORKSPACE / "ternary_checkpoints/run_L3_v8_bin2_P16777216_lr0.001_l20.03_bs128_arr0_t5792.pt",
    262_144: WORKSPACE / "ternary_checkpoints/run_L3_v8_bin2_P16777216_lr0.001_l20.03_bs128_arr0_t262144.pt",
    1_048_576: WORKSPACE / "ternary_checkpoints/run_L3_v8_bin2_P16777216_lr0.001_l20.03_bs128_arr0_t1048576.pt",
}

FINAL_PROBES = {
    "second_last_level1": {
        "row": 0, "col": 0, "level": 1,
        "node": LEVEL1_NODE, "boundary": LEVEL1_BOUNDARY,
        "position": LEVEL1_POSITION,
        "train_targets": level1_train_targets,
        "test_targets": level1_test_targets,
    },
    "second_last_level2": {
        "row": 0, "col": 1, "level": 2,
        "node": LEVEL2_NODE, "boundary": LEVEL2_BOUNDARY,
        "position": LEVEL2_POSITION,
        "train_targets": level2_train_targets,
        "test_targets": level2_test_targets,
    },
    "third_last_level1": {
        "row": 1, "col": 0, "level": 1,
        "node": THIRD_LAST_LEVEL1_NODE, "boundary": THIRD_LAST_LEVEL1_BOUNDARY,
        "position": THIRD_LAST_LEVEL1_POSITION,
        "train_targets": third_level1_train_targets,
        "test_targets": third_level1_test_targets,
    },
    "third_last_level2": {
        "row": 1, "col": 1, "level": 2,
        "node": THIRD_LAST_LEVEL2_NODE, "boundary": THIRD_LAST_LEVEL2_BOUNDARY,
        "position": THIRD_LAST_LEVEL2_POSITION,
        "train_targets": third_level2_train_targets,
        "test_targets": third_level2_test_targets,
    },
}

missing = [str(path) for path in FINAL_CHECKPOINTS.values() if not path.exists()]
assert not missing, "Missing checkpoints:\n" + "\n".join(missing)


@torch.inference_mode()
def extract_multiple_position_activations(model, inputs, positions, batch_size=256):
    """Extract several causal positions in one forward pass.

    Returns a dict mapping each probe name to an array shaped
    (embedding + blocks, examples, hidden_dim).
    """
    for name, position in positions.items():
        if not 0 <= position < inputs.shape[1]:
            raise ValueError(f"{name}: position {position} outside length {inputs.shape[1]}")

    collected = {
        name: [[] for _ in range(model.num_layers + 1)]
        for name in positions
    }

    for start in range(0, len(inputs), batch_size):
        batch = inputs[start:start + batch_size].to(DEVICE)
        captured = {name: {} for name in positions}

        def capture_embedding(module, args):
            embedding = args[0]
            for name, position in positions.items():
                captured[name][0] = embedding[:, position].detach().cpu()

        pre_hook = model.blocks[0].register_forward_pre_hook(capture_embedding)

        def capture_block(layer_index):
            def hook(module, args, output):
                for name, position in positions.items():
                    captured[name][layer_index + 1] = output[:, position].detach().cpu()
            return hook

        block_hooks = [
            block.register_forward_hook(capture_block(layer_index))
            for layer_index, block in enumerate(model.blocks)
        ]
        try:
            model(batch)
        finally:
            pre_hook.remove()
            for hook in block_hooks:
                hook.remove()

        for name in positions:
            for layer_index in range(model.num_layers + 1):
                collected[name][layer_index].append(captured[name][layer_index])

    return {
        name: np.stack([
            torch.cat(layer_batches, dim=0).numpy()
            for layer_batches in collected[name]
        ])
        for name in positions
    }


positions = {name: config["position"] for name, config in FINAL_PROBES.items()}
final_results = {}
final_architecture = None

for requested_step, checkpoint_path in FINAL_CHECKPOINTS.items():
    print(f"\n{'=' * 72}\nCheckpoint t={requested_step}: {checkpoint_path.name}")
    checkpoint_model, checkpoint_arch, saved_step, saved_state = load_model(
        checkpoint_path, NUM_HEADS, DEVICE
    )
    if saved_step is not None:
        assert int(saved_step) == requested_step, (
            f"Filename requests t={requested_step}, checkpoint stores step={saved_step}"
        )
    if final_architecture is None:
        final_architecture = checkpoint_arch
    else:
        assert checkpoint_arch == final_architecture, "Checkpoint architectures differ"

    print("Extracting train activations at all four positions ...")
    train_acts = extract_multiple_position_activations(
        checkpoint_model, train_inputs, positions, ACTIVATION_BATCH_SIZE
    )
    print("Extracting test activations at all four positions ...")
    test_acts = extract_multiple_position_activations(
        checkpoint_model, test_inputs, positions, ACTIVATION_BATCH_SIZE
    )

    final_results[requested_step] = {
        "checkpoint": str(checkpoint_path),
        "saved_step": saved_step,
        "val_loss": saved_state.get("val_loss"),
        "probes": {},
    }
    for probe_name, config in FINAL_PROBES.items():
        print(f"\n{probe_name}")
        final_results[requested_step]["probes"][probe_name] = fit_layerwise_probes(
            train_acts[probe_name], config["train_targets"],
            test_acts[probe_name], config["test_targets"],
            C=PROBE_C,
        )

    del train_acts, test_acts, checkpoint_model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE == "mps":
        torch.mps.empty_cache()


final_layer_labels = ["emb"] + [
    f"L{i}" for i in range(1, final_architecture["num_layers"] + 1)
]

# Keep targets out of the saved metadata; the bundle contains everything needed
# to reproduce the figure without retaining the large generated datasets.
saved_probe_metadata = {
    name: {key: value for key, value in config.items()
           if key not in {"train_targets", "test_targets"}}
    for name, config in FINAL_PROBES.items()
}
final_plot_data = {
    "results": final_results,
    "probe_metadata": saved_probe_metadata,
    "layer_labels": final_layer_labels,
    "chance_accuracy": 1 / V,
    "architecture": final_architecture,
    "grammar": {"V": V, "L": L, "BIN": BIN, "SEED_RULES": SEED_RULES},
    "dataset": {
        "n_train": N_TRAIN, "n_test": N_TEST,
        "train_seed": TRAIN_SEED, "test_seed": TEST_SEED,
        "full_length": D_MAX, "input_length": CONTEXT_LENGTH,
    },
    "probe_C": PROBE_C,
}

RESULTS_PATH = WORKSPACE / "second_third_last_probe_checkpoint_results_ternary_checkpoints.pkl"
with RESULTS_PATH.open("wb") as handle:
    pickle.dump(final_plot_data, handle)
print(f"\nSaved plot data to {RESULTS_PATH}")


# 2×2 layout: rows select second-/third-last; columns select level 1/2.
panel_titles = {
    "second_last_level1": "Second-last level-1 latent (token 24)",
    "second_last_level2": "Second-last level-2 latent (token 18)",
    "third_last_level1": "Third-last level-1 latent (token 21)",
    "third_last_level2": "Third-last level-2 latent (token 9)",
}
colors = {5_792: "tab:blue", 262_144: "tab:orange", 1_048_576: "tab:green"}
x = np.arange(len(final_layer_labels))
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)

for probe_name, config in FINAL_PROBES.items():
    ax = axes[config["row"], config["col"]]
    for step in FINAL_CHECKPOINTS:
        accuracy = final_results[step]["probes"][probe_name]["test"]
        ax.plot(x, accuracy, "o-", linewidth=2.2, markersize=6,
                color=colors[step], label=f"t={step:,}")
    ax.axhline(1 / V, color="gray", linestyle=":", linewidth=1.5,
               label=f"chance = 1/{V}")
    ax.set_title(panel_titles[probe_name])
    ax.set_xticks(x)
    ax.set_xticklabels(final_layer_labels)
    ax.set_ylim(0, 1.02)
    ax.grid(alpha=0.25)

for ax in axes[:, 0]:
    ax.set_ylabel("Held-out linear-probe accuracy")
for ax in axes[1, :]:
    ax.set_xlabel("Probe point")

# One shared legend keeps all four panels visually comparable.
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4, frameon=False,
           bbox_to_anchor=(0.5, 0.99))
fig.suptitle("Second-last and third-last latent information across training",
             fontsize=15, y=1.02)
fig.tight_layout(rect=(0, 0, 1, 0.94))

PNG_PATH = WORKSPACE / "second_third_last_probe_checkpoint_comparison_ternary_checkpoints.png"
PDF_PATH = WORKSPACE / "second_third_last_probe_checkpoint_comparison_ternary_checkpoints.pdf"
fig.savefig(PNG_PATH, dpi=200, bbox_inches="tight")
fig.savefig(PDF_PATH, bbox_inches="tight")
print(f"Saved figure to {PNG_PATH}")
print(f"Saved figure to {PDF_PATH}")
plt.show()

## Compute the replacement Stage 2 only
Run the next cell once before the two standalone plots. It computes only the three displayed latent probes at **t=16384**, at every layer, using the original saved grammar, independent train/test seeds (42/43), sample counts (22000/10000), and classifier settings. Existing checkpoint results are preserved. Later runs reuse the saved replacement file.


In [ ]:
# Self-contained after a kernel restart; no earlier experiment cells needed.
from pathlib import Path
import runpy
runpy.run_path(str(Path.cwd() / "scripts" / "probe_stage2_16384.py"), run_name="__main__")


## Standalone four-stage accuracy bar plot
Run only the next cell to load `second_third_last_probe_checkpoint_results.pkl` and reproduce panel (c) with the revised legend. The 3.4 × 2.8 inch PDF/PNG is sized to sit beside a sketch in a double-column figure. The dotted line marks chance accuracy.
Stage 2 uses **t=16384**, loaded from `second_third_last_probe_step_16384.pkl`; the other stages reuse the original bundle.


In [ ]:
# Standalone: run this cell after a kernel restart; no probe fitting needed.
from pathlib import Path
import pickle
import numpy as np
import matplotlib.pyplot as plt

# Run from the notebook directory, or set DATA_DIR to the results directory.
DATA_DIR = Path.cwd()
RESULTS_PATH = DATA_DIR / "second_third_last_probe_checkpoint_results.pkl"
with RESULTS_PATH.open("rb") as handle:
    plot_data = pickle.load(handle)

# Stage 2 was moved to t=16384. Check that the new probes use the same experiment.
STAGE2_PATH = DATA_DIR / "second_third_last_probe_step_16384.pkl"
if not STAGE2_PATH.exists():
    raise FileNotFoundError(
        f"Missing {STAGE2_PATH}. Run the Stage 2 computation cell above once."
    )
with STAGE2_PATH.open("rb") as handle:
    stage2_data = pickle.load(handle)
for key in ("grammar", "dataset", "architecture", "probe_C", "layer_labels", "chance_accuracy"):
    if stage2_data[key] != plot_data[key]:
        raise ValueError(f"Stage 2 results have different {key}")
for name, metadata in stage2_data["probe_metadata"].items():
    if metadata != plot_data["probe_metadata"][name]:
        raise ValueError(f"Stage 2 probe definition differs: {name}")
plot_data["results"] = dict(plot_data["results"])
plot_data["results"][16384] = stage2_data["results"][16384]

STAGE_STEPS = [512, 16_384, 262_144, 1_048_576]
STAGE_LABELS = ["Stage 1a", "Stage 1b", "Stage 2", "Stage 3"]
PROBES = {
    "second_last_level1": (r"$\mu^{(2)}_8$", "tab:purple"),
    "third_last_level1": (r"$\mu^{(2)}_7$", "tab:brown"),
    "third_last_level2": (r"$\mu^{(1)}_1$", "tab:cyan"),
}
l3_index = plot_data["layer_labels"].index("L3")
accuracies = {
    name: np.array([
        plot_data["results"][step]["probes"][name]["test"][l3_index]
        for step in STAGE_STEPS
    ], dtype=float)
    for name in PROBES
}

# One half of a ~7-inch double-column figure, allowing a gap for the sketch.
FIGSIZE = (3.4, 2.8)  # inches; adjust to the publication's column dimensions.
with plt.rc_context({
    "font.size": 8, "axes.labelsize": 9, "legend.fontsize": 8,
    "xtick.labelsize": 8, "ytick.labelsize": 8,
    "pdf.fonttype": 42, "ps.fonttype": 42,
}):
    fig, ax = plt.subplots(figsize=FIGSIZE)
    x = np.arange(len(STAGE_STEPS))
    bar_width = 0.23
    for i, (name, (label, color)) in enumerate(PROBES.items()):
        ax.bar(x + (i - 1) * bar_width, accuracies[name],
               width=bar_width, color=color, label=label)

    ax.axhline(plot_data["chance_accuracy"], color="gray",
               linestyle=":", linewidth=1.0)
    ax.set(xticks=x, xticklabels=STAGE_LABELS, ylim=(0, 1.02),
           xlabel="Learning stage", ylabel=r"Probe accuracy $\mathcal{A}_Z^{(k,t)}$")
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.2)
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.02),
              ncol=3, fontsize=11, columnspacing=1.2, handletextpad=0.5,
              frameon=False, borderaxespad=0)
    # Fixed margins preserve the requested physical dimensions in both exports.
    fig.subplots_adjust(left=0.17, right=0.98, bottom=0.17, top=0.82)
    for extension in ("pdf", "png"):
        output_path = DATA_DIR / f"latent_probe_accuracy_four_stages.{extension}"
        fig.savefig(output_path, dpi=300)
        print(f"Saved {output_path}")
    plt.show()


## Standalone three-panel summary from saved results
Run only the following cell, including after a kernel restart. It loads `second_third_last_probe_checkpoint_results.pkl` and plots held-out accuracy versus transformer depth for each latent separately: (a) $\mu^{(2)}_8$ (second-last level-1), (b) $\mu^{(2)}_7$ (third-last level-1), and (c) $\mu^{(1)}_1$ (third-last level-2). Each panel shows the same four training stages; the dotted line marks chance accuracy. No analysis needs to be rerun. The cell saves `latent_probe_three_panel_summary.pdf` and `.png` in `DATA_DIR`, replacing existing exports.

Stage 2 uses **t=16384**, loaded from `second_third_last_probe_step_16384.pkl`; the other stages reuse the original bundle.


In [ ]:
# Standalone: run only this cell after restarting the kernel.
from pathlib import Path
import pickle
import numpy as np
import matplotlib.pyplot as plt

# Set DATA_DIR explicitly if the kernel starts outside the notebook directory.
DATA_DIR = Path.cwd()
RESULTS_PATH = DATA_DIR / "second_third_last_probe_checkpoint_results.pkl"
with RESULTS_PATH.open("rb") as handle:
    plot_data = pickle.load(handle)

# Stage 2 was moved to t=16384. Check that the new probes use the same experiment.
STAGE2_PATH = DATA_DIR / "second_third_last_probe_step_16384.pkl"
if not STAGE2_PATH.exists():
    raise FileNotFoundError(
        f"Missing {STAGE2_PATH}. Run the Stage 2 computation cell above once."
    )
with STAGE2_PATH.open("rb") as handle:
    stage2_data = pickle.load(handle)
for key in ("grammar", "dataset", "architecture", "probe_C", "layer_labels", "chance_accuracy"):
    if stage2_data[key] != plot_data[key]:
        raise ValueError(f"Stage 2 results have different {key}")
for name, metadata in stage2_data["probe_metadata"].items():
    if metadata != plot_data["probe_metadata"][name]:
        raise ValueError(f"Stage 2 probe definition differs: {name}")
plot_data["results"] = dict(plot_data["results"])
plot_data["results"][16384] = stage2_data["results"][16384]

final_results = plot_data["results"]
final_layer_labels = plot_data["layer_labels"]
chance_accuracy = plot_data["chance_accuracy"]
print(f"Loaded plot data from {RESULTS_PATH}")

STAGE_STEPS = [512, 16_384, 262_144, 1_048_576]
STAGE_LABELS = ["Stage 1a", "Stage 1b", "Stage 2", "Stage 3"]
STEP_COLORS = {
    512: "tab:purple",
    16_384: "tab:blue",
    262_144: "tab:orange",
    1_048_576: "tab:green",
}
# Paper notation uses root-based depth and one-based node indices:
# level-1 node 7 -> mu^(2)_8; level-1 node 6 -> mu^(2)_7;
# level-2 node 0 -> mu^(1)_1.
PANELS = [
    ("second_last_level1", r"(a) $\mu^{(2)}_8$"),
    ("third_last_level1", r"(b) $\mu^{(2)}_7$"),
    ("third_last_level2", r"(c) $\mu^{(1)}_1$"),
]

x_depth = np.arange(len(final_layer_labels))
fig, axes = plt.subplots(1, 3, figsize=(14.0, 4.2), sharex=True, sharey=True)

for ax, (probe_name, title) in zip(axes, PANELS):
    for stage, step in enumerate(STAGE_STEPS, start=1):
        accuracy = final_results[step]["probes"][probe_name]["test"]
        ax.plot(
            x_depth, accuracy, "o-", color=STEP_COLORS[step],
            linewidth=2.0, markersize=5, label=STAGE_LABELS[stage - 1],
        )
    ax.axhline(chance_accuracy, color="gray", linestyle=":", linewidth=1.3)
    ax.set(
        xticks=x_depth, xticklabels=final_layer_labels,
        xlabel="Probed layer", ylim=(0, 1.02),
    )
    ax.set_title(title, fontsize=22, pad=12)
    ax.xaxis.label.set_size(16)
    ax.tick_params(axis="both", labelsize=13)
    ax.grid(alpha=0.2)

axes[0].set_ylabel("Held-out probe accuracy", fontsize=16)
axes[0].legend(loc="lower right", frameon=False, fontsize=13)
fig.tight_layout(w_pad=1.6)

PDF_PATH = DATA_DIR / "latent_probe_three_panel_summary.pdf"
PNG_PATH = DATA_DIR / "latent_probe_three_panel_summary.png"
fig.savefig(PDF_PATH, bbox_inches="tight")
fig.savefig(PNG_PATH, dpi=300, bbox_inches="tight")
print(f"Saved figure to {PDF_PATH}")
print(f"Saved figure to {PNG_PATH}")
plt.show()
